# Medical Dataset NER: Rule-Based System

## Step-by-Step Process

### Step 1: Data Loading
Two evaluation sets are supported:

- **In-distribution (indist)**: loaded from `MedNER/data/test.jsonl`, which is pre-chunked GLiNER2 JSONL produced by the shared data preparation pipeline. No re-chunking is applied.
- **Out-of-distribution (outdist)**: loaded from `MedNER/151-eval.json` (Label Studio export). Each document is split into overlapping chunks of **1500 characters** with a **300-character overlap**, breaking at sentence boundaries where possible. Entities that straddle a chunk boundary are dropped from that chunk's ground truth.

### Step 2: Negative Sampling (optional)
Controlled by `APPLY_DOWNSAMPLING`. When enabled, entity-free chunks are downsampled to 15% of their original count while all entity-containing chunks are kept. Disabled by default for evaluation.

### Step 3: spaCy Processing
Each chunk is passed through **spaCy** (`en_core_web_sm`) for:
- **Tokenisation**: split text into tokens
- **POS tagging**: assign part-of-speech tags (PROPN, NOUN, ADJ, NUM, PUNCT)
- **Sentence segmentation**: split the chunk into individual sentences

spaCy's built-in NER is disabled; entity detection is handled entirely by the rule-based tracks below.

### Step 4: Sentence Keyword Filter
For each sentence in the chunk, a keyword regex is checked against the following terms:
```
dataset / database / corpus / corpora / treebank / collection / benchmark /
cohort / biobank / registry / repository / challenge / competition /
training set / test set / validation set / trainval
```
Sentences that do not contain any of these terms are skipped. This acts as a precision gate, preventing downstream patterns from firing on unrelated sentences.

### Step 5: spaCy Matcher (POS-based patterns) on filtered sentences
On each sentence that passed the keyword filter, the following spaCy patterns are applied:

| Pattern | What it catches | Example |
|---------|----------------|---------|
| **P1** | Proper-noun sequence ending in a dataset keyword | `PhysioNet SHAREE database` |
| **P2** | Name + train/test/val + set/data | `ImageNet train set` |
| **P3** | ALLCAPS acronym + keyword | `MIMIC dataset`, `CT-ORG corpus` |
| **P4** | Name with embedded year + keyword | `BraTS2021 dataset`, `ISLES22 database` |
| **P5** | Name + 4-digit year + keyword | `BraTS 2021 challenge` |
| **P6** | CamelCase compound name + keyword | `TotalSegmentator dataset` |
| **P7** | Named challenge or competition (no anchor keyword required) | `BraTS Challenge`, `HECKTOR 2021 Challenge` |
| **P8** | Proper noun + parenthesised abbreviation + keyword | `Gene Expression Omnibus (GEO) database` |
| **P_2CHAR_KW** | 2-char ALLCAPS prefix + longer name + keyword | `UK Biobank dataset` |
| **P_2CHAR_COHORT** | 2-char ALLCAPS prefix + cohort/registry/biobank anchor | `UK Biobank cohort` |
| **P_HYPH3_KW** | Hyphenated name with first segment >= 3 chars + keyword | `MIT-BIH dataset` |

The final anchor token for P1-P6 is restricted to a narrow keyword list (`dataset, database, corpus, corpora, treebank, collection, benchmark, trainval`). Broader context words such as `cohort`, `challenge`, and `registry` are kept only in the sentence filter because using them as anchor tokens causes high false-positive rates in medical text.

### Step 6: Document-Wide Regex Tracks (no sentence filter)
Three regex patterns are applied to the full chunk text, bypassing the sentence filter because these forms are structurally unambiguous:

- **Bioinformatics accession IDs**: GSE*, GPL*, TCGA-*, E-MTAB-*, SRP*, EGAS*, phs* (GEO, TCGA, ArrayExpress, SRA, EGA, dbGaP)
- **ALLCAPS hyphenated codes** (first segment >= 4 chars): MIMIC-CXR, LIDC-IDRI, CBIS-DDSM
- **Digit-prefixed hyphenated names**: 3D-IRCADb-01

Within keyword-gated sentences, three additional regex tracks run:

- **Underscore-separated ALLCAPS names**: DA_DRIVE, CHASE_DB1
- **3-4-char ALLCAPS prefix + CamelCase suffix**: CVC-ClinicDB, BraTS-Africa
- **CamelCase-hyphenated names**: BraTS-GLI, KvasirCapsule-SEG

### Step 7: Post-processing
1. **Remove leading articles**: `a` / `an` stripped from the start of a span (`a MIMIC dataset` becomes `MIMIC dataset`)
2. **Resolve overlapping spans**: when two detected spans overlap, the longest span is kept

### Step 8: Evaluation
Predicted entity strings are compared against ground-truth entity strings (both lowercased and stripped):
- **Exact match**: predicted string equals ground-truth string
- **Partial match**: predicted string is a substring of a ground-truth string, or vice versa

Metrics reported: Precision, Recall, F1 for both match types, plus TP, FP, FN counts.


In [1]:
# !pip install -q spacy seqeval nervaluate
# !python -m spacy download en_core_web_sm

In [2]:
import json
import re
import random
import spacy
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from spacy.matcher import Matcher

print('Imports OK')


Imports OK


## Configuration

In [3]:
# DATA MODE:
#   'indist'  -> load MedNER/data/test.jsonl (pre-chunked JSONL), evaluate
#   'outdist' -> load 151-eval.json (Label Studio JSON), chunk it, evaluate
DATA_MODE = 'outdist'

INDIST_PATH  = '../data/test.jsonl'   # pre-chunked GLiNER2 JSONL
OUTDIST_PATH = '../151-eval.json'     # Label Studio JSON (OOD)
SPACY_MODEL  = 'en_core_web_sm'

# Chunking — only used for outdist (indist is already chunked)
CHUNK_SIZE    = 1500
CHUNK_OVERLAP = 300

# Downsampling — mirrors training distribution:
# keep all positive chunks, keep NEGATIVE_SAMPLE_RATIO of entity-free chunks
APPLY_DOWNSAMPLING    = False
NEGATIVE_SAMPLE_RATIO = 0.15
SEED                  = 42

print(f'Mode: {DATA_MODE}  |  downsample: {APPLY_DOWNSAMPLING} (ratio={NEGATIVE_SAMPLE_RATIO})')


Mode: outdist  |  downsample: False (ratio=0.15)


## Chunking & Negative Sampling

In [4]:
def chunk_document(text, entities, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """
    Split document text into overlapping chunks.
    Entities that straddle a chunk boundary are dropped.
    Returns list of {input, output: {entities: {Dataset: [...]}}, has_entities}
    """
    text_len = len(text)
    chunks   = []
    start    = 0

    while start < text_len:
        end = min(start + chunk_size, text_len)

        # Align end to sentence boundary
        if end < text_len:
            search_start = max(start, end - 200)
            for sep in ['. ', '.\n', '\n\n', '\n', ' ']:
                last_sep = text[search_start:end].rfind(sep)
                if last_sep != -1:
                    end = search_start + last_sep + len(sep)
                    break

        chunk_text = text[start:end]
        chunk_ents = [
            e['text'] for e in entities
            if e['start'] >= start and e['end'] <= end
        ]

        chunks.append({
            'input':        chunk_text,
            'output':       {'entities': {'Dataset': chunk_ents}},
            'has_entities': len(chunk_ents) > 0
        })

        if end >= text_len:
            break
        start = end - chunk_overlap

    return chunks


def apply_downsampling(chunks, ratio=NEGATIVE_SAMPLE_RATIO, seed=SEED):
    """
    Keep all positive chunks (has_entities=True).
    Randomly keep `ratio` of negative chunks.
    """
    rng      = random.Random(seed)
    positive = [c for c in chunks if c['has_entities']]
    negative = [c for c in chunks if not c['has_entities']]

    n_keep      = max(1, int(len(negative) * ratio))
    sampled_neg = rng.sample(negative, min(n_keep, len(negative)))

    balanced = positive + sampled_neg
    rng.shuffle(balanced)

    print(f'  positives: {len(positive)}')
    print(f'  negatives: {len(negative)} total  ->  kept {len(sampled_neg)}')
    print(f'  total after downsampling: {len(balanced)}')
    return balanced


print('Chunking & downsampling OK')


Chunking & downsampling OK


## Data Loading

In [5]:
def normalize_text(text):
    """Replace newlines/tabs with spaces (1-to-1, preserves char offsets)."""
    return text.replace('\n', ' ').replace('\t', ' ')


def load_jsonl_indist(path):
    """Load pre-chunked GLiNER2 JSONL — no re-chunking needed."""
    chunks = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec  = json.loads(line)
            ents = rec.get('output', {}).get('entities', {}).get('Dataset', [])
            chunks.append({
                'input':        rec['input'],
                'output':       {'entities': {'Dataset': ents}},
                'has_entities': len(ents) > 0,
            })
    return chunks


def load_indist(path):
    """Load Label Studio JSON -> chunk each document -> return flat chunk list."""
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    all_chunks = []
    for item in data:
        text = normalize_text(item['data']['text'])
        entities = []
        for ann in item.get('annotations', []):
            if ann.get('was_cancelled', False):
                continue
            for result in ann.get('result') or []:
                if result.get('type') == 'labels':
                    v = result['value']
                    s, e = v['start'], v['end']
                    if 0 <= s < e <= len(text):
                        entities.append({'start': s, 'end': e, 'text': text[s:e]})
            break  # first non-cancelled annotation
        all_chunks.extend(chunk_document(text, entities))
    return all_chunks


def load_outdist(path):
    """Load Label Studio JSON (151-eval.json) -> chunk each document -> return flat chunk list."""
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    all_chunks = []
    for item in data:
        text = normalize_text(item['data'].get('text') or item['data'].get('context', ''))
        entities = []
        for ann in item.get('annotations', []):
            if ann.get('was_cancelled', False):
                continue
            for result in ann.get('result') or []:
                if result.get('type') == 'labels':
                    v = result['value']
                    s, e = v['start'], v['end']
                    if 0 <= s < e <= len(text):
                        entities.append({'start': s, 'end': e, 'text': text[s:e]})
            break  # first non-cancelled annotation only
        all_chunks.extend(chunk_document(text, entities))
    return all_chunks


# Quick check of active data source
if DATA_MODE == 'indist':
    _check = load_jsonl_indist(INDIST_PATH)
    print(f'[indist JSONL] {len(_check)} pre-chunked examples')
else:
    _check = load_outdist(OUTDIST_PATH)
    print(f'[outdist] {len(_check)} chunks')

pos = sum(1 for c in _check if c['has_entities'])
tot = sum(len(c['output']['entities']['Dataset']) for c in _check)
print(f'  positive chunks: {pos}  |  negative: {len(_check)-pos}')
print(f'  total golden entities: {tot}')

print('\nData loaders OK')


[outdist] 6178 chunks
  positive chunks: 1540  |  negative: 4638
  total golden entities: 3945

Data loaders OK


## NLP Setup

In [6]:
nlp = spacy.load(SPACY_MODEL, disable=['ner'])
nlp.max_length = 2_000_000
print(f'spaCy: {SPACY_MODEL}')


spaCy: en_core_web_sm


## Keyword Filter & Matcher Patterns

In [7]:
SENTENCE_KEYWORD_RE = re.compile(
    r'\b(?:'
    r'data\s*sets?|data\s*bases?|datasets?|databases?'
    r'|corpus|corpora'
    r'|treebank|tree\s+bank'
    r'|collections?'
    r'|benchmarks?'
    r'|cohorts?'
    r'|biobank'
    r'|registr(?:y|ies)'
    r'|repositor(?:y|ies)'
    r'|challenges?'
    r'|competitions?'
    r'|train(?:ing)?\s+(?:sets?|data|splits?)'
    r'|test(?:ing)?\s+(?:sets?|data|splits?)'
    r'|val(?:idation)?\s+(?:sets?|data|splits?)'
    r'|trainval'
    r')\b',
    re.IGNORECASE
)

KW = (
    r'(?i)^('
    r'data\s*sets?|data\s*bases?|datasets?|databases?'
    r'|corpus|corpora'
    r'|treebank|tree\s*bank'
    r'|collections?'
    r'|benchmarks?'
    r'|cohorts?'
    r'|biobank'
    r'|trainval'
    r')$'
)

matcher = Matcher(nlp.vocab)

# P1 — proper-noun sequence + keyword.
# Anchor POS is PROPN or NOUN only (not ADJ).
#   Reason: spaCy tags "External", "Internal", "Public", "Private" as ADJ.
#   Allowing ADJ as anchor causes "External dataset", "Internal dataset",
#   "Public cohort" etc. to fire — these describe dataset provenance, not names.
#   PROPN covers: RSNA, PhysioNet, BUSI, HAM10000 (named as proper noun)
#   NOUN  covers: CheXpert, Camelyon16, TotalSegmentator (spaCy often tags
#                 novel compound words as NOUN rather than PROPN)
# Middle tokens still allow ADJ (e.g. "Brain Tumor Classification MRI dataset").
# Anchor LENGTH>=3: allows NIH, CXR, GEO; blocks CT, MR, US (2-char).
matcher.add('P1_PROPN_KW', [[
    {'IS_PUNCT': False, 'POS': {'IN': ['PROPN', 'NOUN', 'NUM']}, 'OP': '?'},
    {'LENGTH': {'>=': 3}, 'IS_LOWER': False, 'IS_PUNCT': False, 'IS_DIGIT': False,
     'POS': {'IN': ['PROPN', 'NOUN']}},
    {'IS_PUNCT': False,
     'POS': {'IN': ['PROPN', 'NOUN', 'NUM', 'ADJ']}, 'OP': '{0,4}'},
    {'TEXT': {'REGEX': KW}}
]])

# P2 — same anchor rules + train/test/val split token pair
matcher.add('P2_TWO_PART', [[
    {'IS_PUNCT': False, 'POS': {'IN': ['PROPN', 'NOUN', 'NUM']}, 'OP': '?'},
    {'LENGTH': {'>=': 3}, 'IS_LOWER': False, 'IS_PUNCT': False, 'IS_DIGIT': False,
     'POS': {'IN': ['PROPN', 'NOUN']}},
    {'IS_PUNCT': False,
     'POS': {'IN': ['PROPN', 'NOUN', 'NUM', 'ADJ']}, 'OP': '{0,4}'},
    {'TEXT': {'REGEX': r'(?i)^(train(?:ing)?|test(?:ing)?|val(?:idation)?|eval(?:uation)?|trainval)$'}},
    {'TEXT': {'REGEX': r'(?i)^(sets?|splits?|data(?:set|base)?s?)$'}}
]])

# P3 — ALLCAPS acronym (>=3 chars) + keyword.
matcher.add('P3_ALLCAPS_KW', [[
    {'TEXT': {'REGEX': r'^[A-Z]{2,}(?:[-][A-Za-z0-9]+)*(?:\d{1,8})?$'},
     'LENGTH': {'>=': 3}},
    {'TEXT': {'REGEX': KW}}
]])

# P4 — embedded-year name + required keyword  (BraTS2021 dataset)
matcher.add('P4_EMBEDDED_YEAR', [[
    {'TEXT': {'REGEX': r'^(?:[A-Za-z]{2,})-?\d{2,4}(?:-[A-Za-z0-9]+)*$'}, 'IS_LOWER': False},
    {'TEXT': {'REGEX': KW}}
]])

# P5 — proper noun + 4-digit year + required keyword  (BraTS 2021 dataset)
matcher.add('P5_NAME_YEAR', [[
    {'IS_LOWER': False, 'POS': {'IN': ['PROPN']}, 'LENGTH': {'>=': 3}},
    {'POS': {'IN': ['PROPN', 'NOUN', 'ADJ']}, 'OP': '{0,3}'},
    {'TEXT': {'REGEX': r'^\d{4}$'}},
    {'TEXT': {'REGEX': KW}}
]])

# P6 — CamelCase compound name + required keyword  (TotalSegmentator, VinBigData)
matcher.add('P6_CAMELCASE', [[
    {'TEXT': {'REGEX': r'^[A-Z][a-z]+[A-Z][a-zA-Z0-9]+$'}},
    {'POS': {'IN': ['PROPN', 'NOUN', 'ADJ']}, 'OP': '{0,3}'},
    {'TEXT': {'REGEX': KW}}
]])

# P7 — named challenge / competition
matcher.add('P7_CHALLENGE', [[
    {'IS_LOWER': False, 'POS': {'IN': ['PROPN']}, 'LENGTH': {'>=': 3}},
    {'POS': {'IN': ['PROPN', 'NOUN', 'NUM', 'ADJ']}, 'OP': '{0,4}'},
    {'TEXT': {'REGEX': r'^\d{4}$'}, 'OP': '?'},
    {'LOWER': {'IN': ['challenge', 'competition']}}
]])

# P8 — proper noun + (ABBREVIATION) + required keyword
matcher.add('P8_PAREN_KW', [[
    {'IS_LOWER': False, 'POS': {'IN': ['PROPN', 'NOUN', 'ADJ', 'DET']}, 'LENGTH': {'>=': 2}},
    {'POS': {'IN': ['PROPN', 'NOUN', 'ADJ', 'DET', 'ADP']}, 'OP': '{0,5}'},
    {'TEXT': '('},
    {'TEXT': {'REGEX': r'^[A-Z]{2,8}(?:-[A-Z0-9]+)?$'}},
    {'TEXT': ')'},
    {'TEXT': {'REGEX': KW}}
]])

# Document-wide ALLCAPS hyphenated codes — no sentence filter needed.
# First segment requires 4+ chars (was 3+) to exclude metric abbreviations
# like AUC-ROC (AUC=3), ROC-AUC, CNN-LSTM, MHA-MIL, DEP-MHSA etc.
# MIMIC-CXR, MIMIC-III, CBIS-DDSM, OASIS-3, PHE-SICH-CT-IDS etc.
# These appear in sentences without dataset keywords ("evaluated on MIMIC-CXR"),
# so the sentence filter would silently miss them.
# Requires >=3 uppercase letters in first segment and >=2 in subsequent segments,
# excluding metric strings (AUC-ROC: second segment ROC has 3 chars — is caught,
# so we further exclude by keeping HYPH_RE inside keyword-gated sentences only
# when the match doesn't look like a metric/method abbreviation).
HYPH_RE = re.compile(r'\b([A-Z]{4,}[0-9]*(?:-[A-Z]{2,}[0-9]*)+)\b')

# P_2CHAR_KW — 2-char ALLCAPS prefix + longer name + keyword.
# Catches "UK Biobank dataset", "LA dataset" (left atrium), "GE dataset" etc.
matcher.add('P_2CHAR_KW', [[
    {'IS_UPPER': True, 'LENGTH': 2, 'POS': {'IN': ['PROPN', 'NOUN']}},
    {'IS_LOWER': False, 'IS_PUNCT': False, 'LENGTH': {'>=': 4},
     'POS': {'IN': ['PROPN', 'NOUN']}},
    {'IS_PUNCT': False, 'POS': {'IN': ['PROPN', 'NOUN', 'ADJ']}, 'OP': '{0,3}'},
    {'TEXT': {'REGEX': KW}}
]])

# P_2CHAR_COHORT — 2-char ALLCAPS prefix ending in cohort/registry/biobank.
matcher.add('P_2CHAR_COHORT', [[
    {'IS_UPPER': True, 'LENGTH': 2, 'POS': {'IN': ['PROPN', 'NOUN']}},
    {'IS_LOWER': False, 'IS_PUNCT': False, 'LENGTH': {'>=': 4},
     'POS': {'IN': ['PROPN', 'NOUN']}},
    {'IS_PUNCT': False, 'POS': {'IN': ['PROPN', 'NOUN', 'ADJ']}, 'OP': '{0,3}'},
    {'LOWER': {'IN': ['cohort', 'cohorts', 'registry', 'biobank', 'study', 'initiative']}}
]])

# P_HYPH3_KW — hyphenated name with first segment >=3 chars + keyword.
# Catches "MIT-BIH dataset", "Retinal-OCT2017 dataset" which HYPH_RE misses
# because HYPH_RE requires first segment >=4 chars.
matcher.add('P_HYPH3_KW', [[
    {'IS_LOWER': False, 'IS_DIGIT': False, 'LENGTH': {'>=': 3},
     'POS': {'IN': ['PROPN', 'NOUN', 'ADJ']}},
    {'TEXT': '-'},
    {'IS_LOWER': False, 'LENGTH': {'>=': 2}},
    {'IS_PUNCT': False, 'POS': {'IN': ['PROPN', 'NOUN', 'ADJ']}, 'OP': '{0,2}'},
    {'TEXT': {'REGEX': KW}}
]])

# Digit-prefixed hyphenated names — document-wide regex.
# Catches: 3D-IRCADb-01, 3d-ircadb-01
DIGIT_HYPH_RE = re.compile(r'\b([0-9]+[A-Za-z]{2,}(?:[-][A-Za-z0-9]{2,})+)\b')

# Underscore-separated ALLCAPS names — detected within keyword sentences.
# All segments must be ALLCAPS to block tool names like RF_purify76.
# Catches: DA_DRIVE, HV_NIR, DA_CHASE, CHASE_DB1
UNDER_RE = re.compile(r'\b([A-Z]{2,}[0-9]*(?:_[A-Z]{2,}[0-9]*)+)\b')

# Architecture/method name filter for CamelCase detection.
# Pattern 1 — suffix-based: blocks Net, Former, GRU, LSTM, etc.
#   Also blocks method names ending in LIFT (DeepLIFT), plore (AttExplore),
#   Explore, Purify, Boost, Score, Pred.
CAMEL_ARCH_RE = re.compile(
    r'(?:Net|Former|Encoder|Decoder|GRU|LSTM|Conv|Unet|Vnet|Fuse|Fit|'
    r'CLR|Mix|Mamba|Flow|Gen|Gan|Gnn|Gcn|Rnn|Fcn|SVM|'
    r'LIFT|plore|Explore|Purify|Boost|Score|Pred)$'
)
# Pattern 2 — prefix-based: blocks ResNet, DenseNet, EfficientNet,
#   GradCAM, EvoMDT and similar method/architecture names.
CAMEL_ARCH_PREFIX_RE = re.compile(
    r'^(?:Res|Dense|Mobile|Inception|Squeeze|Alex|Bi[A-Z]|Se[gG]|Trans[A-Z]|'
    r'Grad|Evo|Efficient)'
)

# Blocklist for standalone ALLCAPS tokens that should never be dataset names.
ALLCAPS_BLOCKLIST = {
    # Metrics / statistics
    'dice', 'ssim', 'psnr', 'auroc', 'rmse', 'mape', 'auprc',
    # ML model/method names (standalone uppercase forms)
    'bert', 'clip', 'gelu', 'relu', 'adam', 'unet', 'vnet',
    'wgan', 'cgan', 'dcnn', 'smote', 'sota', 'lstm', 'shap',
    # Model version names
    'vgg16', 'vgg19',
    # Medical imaging modalities/sequences (not dataset names)
    'bold', 'spect', 'dicom', 'fmri', 'cbct', 'flair', 'dce', 'dwi', 'dti',
    # Disease acronyms that are not dataset names in isolation
    'sars',
    # Software features / notation systems
    'autotune', 'smiles',
    # Section headings / document structure words
    'results', 'methods', 'experiments', 'discussion', 'conclusions',
    'introduction', 'related', 'background', 'references', 'appendix',
    'table', 'figure', 'algorithm', 'supplement', 'supplementary',
    'baselines', 'baseline', 'comparisons', 'comparison', 'ablation',
    # Generic words that appear all-caps in PDF-extracted papers
    'note', 'same', 'work', 'best', 'each', 'also', 'both',
    'data', 'base', 'full', 'main', 'long', 'wide', 'high',
    'deep', 'fast', 'next', 'rads', 'state', 'true', 'false',
    # Specific method-name FPs confirmed by error analysis
    'knrm', 'latte', 'dkrnn', 'gafm',
}

# New: 3-4-char ALLCAPS prefix + CamelCase suffix — detected in keyword sentences.
# Catches: CVC-ClinicDB, CVC-ColonDB, BraTS-Africa-ish, SARS-CoV-2 (CoV=C+ov).
# AUC-ROC/CNN-LSTM blocked because their suffix has no lowercase after first char.
SEMI_HYPH_RE = re.compile(
    r'\b([A-Z]{3,4}[0-9]*-[A-Z][a-z][a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*)\b'
)

# New: CamelCase-hyphenated dataset names — detected in keyword sentences.
# Catches: BraTS-GLI, KvasirCapsule-SEG, BraTS-Africa.
# Requires genuine CamelCase before the hyphen (at least one internal uppercase).
CAMEL_HYPH_RE = re.compile(
    r'\b([A-Z][a-z]+[A-Z][a-zA-Z0-9]+-[A-Z][a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*)\b'
)

print(f'Matcher: {len(matcher)} patterns loaded')


Matcher: 11 patterns loaded


In [8]:
# Bioinformatics accession IDs — applied to full chunk text, no sentence filter needed
BIOACC_RE = re.compile(
    r'\b('
    r'GSE\d{4,8}'                 # GEO Series
    r'|GPL\d{4,8}'                # GEO Platform
    r'|GDS\d{4,8}'                # GEO DataSet
    r'|TCGA-[A-Z]{2,6}'           # TCGA project codes
    r'|E-[A-Z]{3,8}-\d{1,6}'     # ArrayExpress
    r'|SRP\d{5,9}'                # SRA Project
    r'|SRX\d{5,9}'                # SRA Experiment
    r'|EGA[SD]\d{8,12}'          # EGA Study/Dataset
    r'|phs\d{6}\.v\d+\.p\d+'    # dbGaP full form
    r'|phs\d{6}'                  # dbGaP short form
    r')\b'
)

print('Bioinformatics accession regex OK')


Bioinformatics accession regex OK


## Prediction

`predict_entities(text)` runs two tracks in parallel on each chunk, then merges and deduplicates:

**Track A — Bioinformatics accession IDs** (no sentence filter, runs on full chunk)
Regex catches structurally unique IDs (GSE*, TCGA-*, E-MTAB-*, SRP*, EGAS*, phs*) that
never appear with a dataset keyword and would be missed by Track B.

**Track B — spaCy Matcher** (runs only on sentences that contain a keyword)
1. `doc.sents` splits the chunk into sentences
2. `SENTENCE_KEYWORD_RE` checks each sentence — skip if no keyword found
3. `matcher(sent_span)` fires all 8 POS-based patterns on the filtered sentence
4. Character offsets are recovered from token positions

**Post-processing** (applied to merged spans from both tracks):
1. Strip leading `a` / `an`
2. Resolve overlapping spans — keep longest


In [9]:
# CamelCase regex — matches dataset names like CheXpert, BraTS, PanNuke
CAMEL_RE = re.compile(r'^[A-Z][a-z]+[A-Z][a-zA-Z0-9]+$')


def remove_leading_articles(text, spans):
    art_re = re.compile(r'^(a|an)\s+', re.IGNORECASE)
    out = []
    for start, end in spans:
        m = art_re.match(text[start:end])
        if m:
            ns = start + m.end()
            if ns < end:
                out.append((ns, end))
                continue
        out.append((start, end))
    return out


def resolve_overlaps(spans):
    if not spans:
        return []
    spans = list(set(spans))
    spans.sort(key=lambda x: (x[0], -(x[1] - x[0])))
    result, max_end = [], -1
    for start, end in spans:
        if start >= max_end:
            result.append((start, end))
            max_end = end
        elif end > max_end and result and start <= result[-1][0]:
            result[-1] = (start, end)
            max_end = end
    return result


def predict_entities(text, mode=DATA_MODE):
    doc = nlp(text)
    char_spans = []

    # Track A — bioinformatics accession IDs (doc-wide)
    for m in BIOACC_RE.finditer(text):
        char_spans.append((m.start(), m.end()))

    # Track B — ALLCAPS hyphenated codes, first segment >=4 chars (doc-wide)
    for m in HYPH_RE.finditer(text):
        char_spans.append((m.start(), m.end()))

    # Track B2 — digit-prefixed hyphenated names (doc-wide)
    for m in DIGIT_HYPH_RE.finditer(text):
        char_spans.append((m.start(), m.end()))

    # Tracks C-H — keyword-sentence gated
    for sent in doc.sents:
        if not SENTENCE_KEYWORD_RE.search(sent.text):
            continue
        sent_span = doc[sent.start:sent.end]

        # Track C: spaCy Matcher (P1-P8 + P_2CHAR_KW + P_2CHAR_COHORT + P_HYPH3_KW)
        seen = set()
        for _, start, end in matcher(sent_span):
            ts = sent_span[start]
            te = sent_span[end - 1]
            cs, ce = ts.idx, te.idx + len(te.text)
            if (cs, ce) not in seen:
                char_spans.append((cs, ce))
                seen.add((cs, ce))

        for tok in sent_span:
            # Track D: CamelCase (architecture/method names filtered via suffix+prefix)
            if (CAMEL_RE.match(tok.text)
                    and len(tok.text) >= 5
                    and tok.pos_ in ('PROPN', 'NOUN')
                    and not CAMEL_ARCH_RE.search(tok.text)
                    and not CAMEL_ARCH_PREFIX_RE.match(tok.text)):
                char_spans.append((tok.idx, tok.idx + len(tok.text)))

            # Track E: standalone ALLCAPS >=4 chars
            # Filtered: blocklist, no dots (blocks ICD codes like M31.6),
            # at least 2 alphabetic chars (blocks single-letter prefixes like M31).
            if (tok.is_upper
                    and len(tok.text) >= 4
                    and tok.pos_ in ('PROPN', 'NOUN', 'X')
                    and not tok.is_stop
                    and tok.lower_ not in ALLCAPS_BLOCKLIST
                    and '.' not in tok.text
                    and sum(c.isalpha() for c in tok.text) >= 2):
                char_spans.append((tok.idx, tok.idx + len(tok.text)))

        # Track F: underscore-separated all-caps names (all-caps segments required)
        for m in UNDER_RE.finditer(sent.text):
            offset = sent_span[0].idx
            char_spans.append((offset + m.start(), offset + m.end()))

        # Track G: 3-4-char ALLCAPS prefix + CamelCase suffix (KW-gated)
        # Catches CVC-ClinicDB, SARS-CoV-2; AUC-ROC/CNN-LSTM blocked (no lowercase
        # after the first char of second segment in those abbreviations).
        for m in SEMI_HYPH_RE.finditer(sent.text):
            offset = sent_span[0].idx
            char_spans.append((offset + m.start(), offset + m.end()))

        # Track H: CamelCase-hyphenated dataset names (KW-gated)
        # Catches BraTS-GLI, KvasirCapsule-SEG (genuine CamelCase before hyphen).
        for m in CAMEL_HYPH_RE.finditer(sent.text):
            offset = sent_span[0].idx
            char_spans.append((offset + m.start(), offset + m.end()))

    char_spans = remove_leading_articles(text, char_spans)
    char_spans = resolve_overlaps(char_spans)
    return [text[s:e].strip() for s, e in char_spans if text[s:e].strip()]


print('Prediction function OK')


Prediction function OK


## Evaluation
- **Exact match** — predicted text (lowercased + stripped) == ground truth
- **Partial match** — predicted text is substring of ground truth or vice versa

In [10]:
def compute_metrics(all_true, all_pred):
    """
    all_true, all_pred: lists of sets of lowercased entity strings (one set per chunk).
    Returns exact and partial precision/recall/F1.
    """
    tp = fp = fn = 0
    partial_tp = 0

    for true_set, pred_set in zip(all_true, all_pred):
        tp += len(true_set & pred_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)

        for pmention in pred_set:
            if pmention not in true_set:
                for tmention in true_set:
                    if pmention in tmention or tmention in pmention:
                        partial_tp += 1
                        break

    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-8)

    partial_precision = (tp + partial_tp) / max(tp + fp, 1)
    partial_recall    = (tp + partial_tp) / max(tp + fn, 1)
    partial_f1        = 2 * partial_precision * partial_recall / max(partial_precision + partial_recall, 1e-8)

    return {
        'exact':   {'precision': precision,         'recall': recall,         'f1': f1,
                    'tp': tp, 'fp': fp, 'fn': fn},
        'partial': {'precision': partial_precision, 'recall': partial_recall, 'f1': partial_f1,
                    'partial_tp': partial_tp},
        'total_true': tp + fn,
        'total_pred': tp + fp,
    }


def evaluate(chunks, mode=DATA_MODE, verbose=True):
    all_true, all_pred = [], []
    for chunk in chunks:
        gold_strs = chunk['output']['entities'].get('Dataset', [])
        true_set  = {m.strip().lower() for m in gold_strs}
        pred_set  = {p.strip().lower() for p in predict_entities(chunk['input'], mode=mode)}
        all_true.append(true_set)
        all_pred.append(pred_set)

    metrics = compute_metrics(all_true, all_pred)

    if verbose:
        e = metrics['exact']
        p = metrics['partial']
        print(f'  Total true entities : {metrics["total_true"]}')
        print(f'  Total predicted     : {metrics["total_pred"]}')
        print()
        hdr = f'{"":16} {"Precision":>10} {"Recall":>10} {"F1":>10}'
        print(hdr)
        print('-' * len(hdr))
        print(f'{"Exact match":<16} {e["precision"]:>10.4f} {e["recall"]:>10.4f} {e["f1"]:>10.4f}')
        print(f'{"Partial match":<16} {p["precision"]:>10.4f} {p["recall"]:>10.4f} {p["f1"]:>10.4f}')
        print()
        print(f'  TP={e["tp"]}  FP={e["fp"]}  FN={e["fn"]}  partial_tp={p["partial_tp"]}')

    return metrics


print('Evaluation functions OK')


Evaluation functions OK


## Error Analysis

In [11]:
def error_analysis(chunks, mode=DATA_MODE, n=20, context_window=80):
    fp_list, fn_list = [], []

    for chunk in chunks:
        text      = chunk['input']
        gold_strs = chunk['output']['entities'].get('Dataset', [])
        true_set  = {m.strip().lower() for m in gold_strs}
        pred_list = predict_entities(text, mode=mode)
        pred_set  = {p.strip().lower() for p in pred_list}

        for p in pred_list:
            pl = p.strip().lower()
            if pl not in true_set and not any(pl in t or t in pl for t in true_set):
                idx = text.lower().find(pl)
                ctx = text[max(0, idx-context_window):idx+len(p)+context_window] if idx != -1 else ''
                fp_list.append({'entity': p, 'context': ctx})

        for g in gold_strs:
            gl = g.strip().lower()
            if gl not in pred_set and not any(gl in p or p in gl for p in pred_set):
                idx = text.lower().find(gl)
                ctx = text[max(0, idx-context_window):idx+len(g)+context_window] if idx != -1 else ''
                fn_list.append({'entity': g, 'context': ctx})

    print(f'\n{"="*60}')
    print(f'FALSE POSITIVES  ({len(fp_list)} total) — top {n}')
    print(f'{"="*60}')
    for fp in fp_list[:n]:
        print(f'  Entity : {repr(fp["entity"])}')
        print(f'  Context: ...{fp["context"]}...')
        print()

    print(f'\n{"="*60}')
    print(f'FALSE NEGATIVES  ({len(fn_list)} total) — top {n}')
    print(f'{"="*60}')
    for fn in fn_list[:n]:
        print(f'  Entity : {repr(fn["entity"])}')
        print(f'  Context: ...{fn["context"]}...')
        print()

    print('Top-15 FP entity texts:')
    for txt, cnt in Counter(fp['entity'].lower() for fp in fp_list).most_common(15):
        print(f'  {cnt:4d}x  {repr(txt)}')

    print('\nTop-15 FN entity texts (missed):')
    for txt, cnt in Counter(fn['entity'].lower() for fn in fn_list).most_common(15):
        print(f'  {cnt:4d}x  {repr(txt)}')

    return fp_list, fn_list


print('Error analysis function OK')


Error analysis function OK


## Run

In [12]:
def run_mode(mode, apply_ds=APPLY_DOWNSAMPLING):
    if mode == 'indist':
        chunks = load_jsonl_indist(INDIST_PATH)
        label  = 'In-Distribution (test.jsonl)'
    else:
        chunks = load_outdist(OUTDIST_PATH)
        label  = 'Out-of-Distribution (151-eval)'

    if apply_ds:
        chunks = apply_downsampling(chunks)
        ds_label = f'downsampled (ratio={NEGATIVE_SAMPLE_RATIO})'
    else:
        ds_label = 'no downsampling'

    print(f'\n{"#"*60}')
    print(f'  RULE-BASED NER -- {label.upper()} | {ds_label}')
    print(f'{"#"*60}\n')

    metrics = evaluate(chunks, mode=mode, verbose=True)
    return metrics, chunks


# ── Run both sets (no downsampling — test.jsonl is already clean) ─────────────
metrics_id,   chunks_id   = run_mode('indist',  apply_ds=False)
metrics_ood,  chunks_ood  = run_mode('outdist', apply_ds=False)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"="*68}')
print('SUMMARY')
print(f'{"="*68}')
print(f'{"":<40} {"ExactF1":>8} {"P":>7} {"R":>7} {"PartialF1":>10}')
print(f'{"-"*68}')
for label, m in [
    ('In-Distribution  (test.jsonl, no ds)  ', metrics_id),
    ('Out-of-Distribution (151-eval, no ds) ', metrics_ood),
]:
    e = m['exact']
    print(f'{label:<40} {e["f1"]:>8.4f} {e["precision"]:>7.4f} {e["recall"]:>7.4f} {m["partial"]["f1"]:>10.4f}')
print(f'{"="*68}')



############################################################
  RULE-BASED NER -- IN-DISTRIBUTION (TEST.JSONL) | no downsampling
############################################################

  Total true entities : 1496
  Total predicted     : 1662

                  Precision     Recall         F1
-------------------------------------------------
Exact match          0.4224     0.4693     0.4446
Partial match        0.6071     0.6745     0.6390

  TP=702  FP=960  FN=794  partial_tp=307

############################################################
  RULE-BASED NER -- OUT-OF-DISTRIBUTION (151-EVAL) | no downsampling
############################################################

  Total true entities : 3043
  Total predicted     : 4507

                  Precision     Recall         F1
-------------------------------------------------
Exact match          0.3838     0.5685     0.4583
Partial match        0.4910     0.7272     0.5862

  TP=1730  FP=2777  FN=1313  partial_tp=483

SUMMARY
  

In [13]:
print('\n' + '='*60)
print('ERROR ANALYSIS -- IN-DISTRIBUTION')
print('='*60)
fp_id, fn_id = error_analysis(chunks_id, n=15)



ERROR ANALYSIS -- IN-DISTRIBUTION

FALSE POSITIVES  (762 total) — top 15
  Entity : 'HMLC'
  Context: ... formulation, respectively. Datasets and Taxonomy The first step in creating an HMLC system is to create the label taxonomy. In this work, our main results focus on...

  Entity : 'AML datasets'
  Context: ...s and of metabolic networks (expected in version 4). We also intend to use more AML datasets (potentially all published ones) and to explore subtypes of this acute leukemia...

  Entity : 'MIMIC-III'
  Context: ...nglish-Wiki [42] MLM+NSP --12 BertLayers Single 77.6 (MedNLI) ClinicalBERT [43] MIMIC-III [44] MLM+NSP --12 BertLayers Single 80.8 (MedNLI) * MLM: Mask Languae Modeling;...

  Entity : 'CBIS-DDSM'
  Context: ...n discriminator are co-trained for domain adaptation. Experiments on the public CBIS-DDSM , InBreast and a self-collected dataset demonstrate its effectiveness. Yang et ...

  Entity : 'ATTENTION-BASED'
  Context: ...ffectiveness of our proposed method. TABLE 

In [14]:
print('\n' + '='*60)
print('ERROR ANALYSIS -- OUT-OF-DISTRIBUTION')
print('='*60)
fp_ood, fn_ood = error_analysis(chunks_ood, n=15)



ERROR ANALYSIS -- OUT-OF-DISTRIBUTION

FALSE POSITIVES  (2854 total) — top 15
  Entity : 'Holter database'
  Context: ...ich contains 24 h ECG recordings of 139 hypertensive patients obtained from the Holter database. Of these patients, 90 were male. The entire processing workflow-including lite...

  Entity : 'Medical Literature Review Database'
  Context: ...RV parameter with vascular outcomes. This literature-derived dataset formed the Medical Literature Review Database (MLRD)—a central reference that served as the foundation for our subsequent fea...

  Entity : 'ChatGPT'
  Context: ...ential bias from pre-trained knowledge, we also included instructions prompting ChatGPT to perform reasoning before making any final decisions. The MLRD was constructe...

  Entity : 'SDNN'
  Context: ...mes before inputting them into the model; for instance, HF, LF/HF, VLF, LF, TP, SDNN, and rMSSD were replaced with the labels A through G. Using this coded format a...

  Entity : 'ECG database'
  C

In [15]:
print(f'\n{"="*60}')
print('FINAL SUMMARY')
print(f'{"="*60}')
for lbl, m in [('In-Distribution  (test.jsonl)', metrics_id),
                ('Out-of-Distribution (151-eval)', metrics_ood)]:
    e = m['exact']; p = m['partial']
    print(f'\n{lbl}')
    print(f'  Exact  : P={e["precision"]:.4f}  R={e["recall"]:.4f}  F1={e["f1"]:.4f}  (TP={e["tp"]} FP={e["fp"]} FN={e["fn"]})')
    print(f'  Partial: P={p["precision"]:.4f}  R={p["recall"]:.4f}  F1={p["f1"]:.4f}')



FINAL SUMMARY

In-Distribution  (test.jsonl)
  Exact  : P=0.4224  R=0.4693  F1=0.4446  (TP=702 FP=960 FN=794)
  Partial: P=0.6071  R=0.6745  F1=0.6390

Out-of-Distribution (151-eval)
  Exact  : P=0.3838  R=0.5685  F1=0.4583  (TP=1730 FP=2777 FN=1313)
  Partial: P=0.4910  R=0.7272  F1=0.5862


## Quick Pattern Tester

In [16]:
def test_sentence(sentence):
    doc = nlp(sentence)
    print('Tokens & POS:')
    for tok in doc:
        print(f'  {tok.text!r:<25} {tok.pos_:<8} lower={tok.is_lower}')
    preds = predict_entities(sentence)
    print('\nPredicted entities:', preds if preds else '(none)')


test_sentence('We trained on the PhysioNet SHAREE database and evaluated on BraTS 2021 challenge.')


Tokens & POS:
  'We'                      PRON     lower=False
  'trained'                 VERB     lower=True
  'on'                      ADP      lower=True
  'the'                     DET      lower=True
  'PhysioNet'               PROPN    lower=False
  'SHAREE'                  NOUN     lower=False
  'database'                NOUN     lower=True
  'and'                     CCONJ    lower=True
  'evaluated'               VERB     lower=True
  'on'                      ADP      lower=True
  'BraTS'                   PROPN    lower=False
  '2021'                    NUM      lower=False
  'challenge'               NOUN     lower=True
  '.'                       PUNCT    lower=False

Predicted entities: ['PhysioNet SHAREE database', 'BraTS 2021 challenge']


In [17]:
test_sentence('Gene Expression Omnibus (GEO) series GSE110554 and GSE112179 were downloaded.')


Tokens & POS:
  'Gene'                    PROPN    lower=False
  'Expression'              PROPN    lower=False
  'Omnibus'                 PROPN    lower=False
  '('                       PUNCT    lower=False
  'GEO'                     PROPN    lower=False
  ')'                       PUNCT    lower=False
  'series'                  PROPN    lower=True
  'GSE110554'               PROPN    lower=False
  'and'                     CCONJ    lower=True
  'GSE112179'               PROPN    lower=False
  'were'                    AUX      lower=True
  'downloaded'              VERB     lower=True
  '.'                       PUNCT    lower=False

Predicted entities: ['GSE110554', 'GSE112179']


In [18]:
test_sentence('TotalSegmentator and AutoPET datasets were used for pre-training the model.')


Tokens & POS:
  'TotalSegmentator'        PROPN    lower=False
  'and'                     CCONJ    lower=True
  'AutoPET'                 ADJ      lower=False
  'datasets'                NOUN     lower=True
  'were'                    AUX      lower=True
  'used'                    VERB     lower=True
  'for'                     ADP      lower=True
  'pre'                     ADJ      lower=True
  '-'                       ADJ      lower=False
  'training'                VERB     lower=True
  'the'                     DET      lower=True
  'model'                   NOUN     lower=True
  '.'                       PUNCT    lower=False

Predicted entities: ['TotalSegmentator', 'AutoPET datasets']


In [19]:
test_sentence('The TCGA-LUAD cohort along with the LIDC-IDRI dataset were used for validation.')


Tokens & POS:
  'The'                     DET      lower=False
  'TCGA'                    PROPN    lower=False
  '-'                       PUNCT    lower=False
  'LUAD'                    PROPN    lower=False
  'cohort'                  VERB     lower=True
  'along'                   ADP      lower=True
  'with'                    ADP      lower=True
  'the'                     DET      lower=True
  'LIDC'                    PROPN    lower=False
  '-'                       PUNCT    lower=False
  'IDRI'                    ADJ      lower=False
  'dataset'                 NOUN     lower=True
  'were'                    AUX      lower=True
  'used'                    VERB     lower=True
  'for'                     ADP      lower=True
  'validation'              NOUN     lower=True
  '.'                       PUNCT    lower=False

Predicted entities: ['TCGA-LUAD cohort', 'LIDC-IDRI dataset']
